# 01 — Dataset Exploration

**Purpose:** Explore the fall detection datasets (URFD, Le2i, UP-Fall) to understand their structure, class distribution, sequence lengths, and keypoint quality before training.

**Reference:** research/1.1 — Dataset Survey and Analysis

**Outputs:**
- Dataset statistics (sequences, subjects, class balance)
- Sequence length distributions
- Keypoint confidence histograms
- Sample skeleton visualizations
- LOSO fold composition

In [ ]:
import sys
from pathlib import Path

# Ensure project root is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

from src.data_processing.loader import DatasetLoader, LABEL_FALL, LABEL_ADL
from src.data_processing.splitter import SubjectSplitter
from src.utils.config import Config

config = Config()
print(f"Project root: {PROJECT_ROOT}")
print(f"Supported datasets: {DatasetLoader.supported_datasets()}")
print(f"Random seed: {config.seed}")

## 1. Load Dataset Metadata

Load whichever datasets are available in `data/raw/`. If none are downloaded yet, the cells below will show instructions.

In [ ]:
def explore_dataset(dataset_name: str) -> dict:
    """Load a dataset and print summary statistics."""
    raw_dir = PROJECT_ROOT / "data" / "raw" / dataset_name
    
    if not raw_dir.exists():
        print(f"  [SKIP] {dataset_name}: directory not found at {raw_dir}")
        print(f"         Run: python scripts/download_dataset.py {dataset_name}")
        return None
    
    try:
        loader = DatasetLoader(dataset_name, raw_dir)
        sequences = loader.load()
    except Exception as e:
        print(f"  [SKIP] {dataset_name}: {e}")
        return None
    
    if not sequences:
        print(f"  [SKIP] {dataset_name}: no sequences found")
        return None
    
    # Compute statistics
    subjects = sorted(set(s.subject_id for s in sequences))
    falls = [s for s in sequences if s.label == LABEL_FALL]
    adls = [s for s in sequences if s.label == LABEL_ADL]
    cameras = sorted(set(s.camera for s in sequences))
    
    stats = {
        "name": dataset_name,
        "total": len(sequences),
        "falls": len(falls),
        "adls": len(adls),
        "subjects": subjects,
        "num_subjects": len(subjects),
        "cameras": cameras,
        "sequences": sequences,
    }
    
    print(f"\n{'='*50}")
    print(f"  Dataset: {dataset_name.upper()}")
    print(f"{'='*50}")
    print(f"  Total sequences:  {stats['total']}")
    print(f"  Fall sequences:   {stats['falls']}")
    print(f"  ADL sequences:    {stats['adls']}")
    print(f"  Subjects:         {stats['num_subjects']} {subjects}")
    print(f"  Cameras:          {cameras}")
    print(f"  Fall ratio:       {stats['falls']/max(stats['total'],1):.1%}")
    
    return stats

# Try loading all supported datasets
all_stats = {}
for ds_name in ["urfd", "le2i", "up_fall"]:
    result = explore_dataset(ds_name)
    if result:
        all_stats[ds_name] = result

if not all_stats:
    print("\nNo datasets found. Download at least URFD to continue:")
    print("  python scripts/download_dataset.py urfd")

## 2. Class Distribution Visualization

Visualize the fall vs. ADL balance per dataset. Class imbalance is critical — research/8.2 mandates automatic class weight balancing.

In [ ]:
if all_stats:
    fig, axes = plt.subplots(1, len(all_stats), figsize=(5 * len(all_stats), 4))
    if len(all_stats) == 1:
        axes = [axes]
    
    for ax, (name, stats) in zip(axes, all_stats.items()):
        counts = [stats["adls"], stats["falls"]]
        labels = ["ADL", "Fall"]
        colors = ["#4CAF50", "#F44336"]
        
        bars = ax.bar(labels, counts, color=colors, edgecolor="black", linewidth=0.5)
        ax.set_title(f"{name.upper()}\n({stats['total']} total)", fontsize=12)
        ax.set_ylabel("Number of sequences")
        
        for bar, count in zip(bars, counts):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                str(count),
                ha="center", va="bottom", fontweight="bold",
            )
    
    plt.suptitle("Class Distribution per Dataset", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No datasets loaded — skipping visualization.")

## 3. Per-Subject Sequence Distribution

Visualize how many sequences each subject has. This is critical for understanding LOSO fold balance — a fold with very few training falls may produce unreliable results.

In [ ]:
if all_stats:
    for name, stats in all_stats.items():
        subjects = stats["subjects"]
        sequences = stats["sequences"]
        
        fall_per_subject = {s: 0 for s in subjects}
        adl_per_subject = {s: 0 for s in subjects}
        
        for seq in sequences:
            if seq.label == LABEL_FALL:
                fall_per_subject[seq.subject_id] += 1
            else:
                adl_per_subject[seq.subject_id] += 1
        
        fig, ax = plt.subplots(figsize=(max(6, len(subjects) * 1.2), 4))
        
        x = np.arange(len(subjects))
        width = 0.35
        
        falls = [fall_per_subject[s] for s in subjects]
        adls = [adl_per_subject[s] for s in subjects]
        
        ax.bar(x - width/2, adls, width, label="ADL", color="#4CAF50", edgecolor="black", linewidth=0.5)
        ax.bar(x + width/2, falls, width, label="Fall", color="#F44336", edgecolor="black", linewidth=0.5)
        
        ax.set_xlabel("Subject ID")
        ax.set_ylabel("Number of sequences")
        ax.set_title(f"{name.upper()} — Sequences per Subject")
        ax.set_xticks(x)
        ax.set_xticklabels(subjects, rotation=45 if len(subjects) > 8 else 0)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n{name.upper()} per-subject summary:")
        for s in subjects:
            print(f"  {s}: {adl_per_subject[s]} ADL, {fall_per_subject[s]} falls")
else:
    print("No datasets loaded — skipping.")

## 4. LOSO Cross-Validation Fold Analysis

Generate LOSO folds and analyze the training/test split for each fold. Per research/5.1, LOSO is mandatory — we must verify each fold has sufficient training data.

In [ ]:
if all_stats:
    for name, stats in all_stats.items():
        subjects = stats["subjects"]
        sequences = stats["sequences"]
        
        splitter = SubjectSplitter(subjects=subjects, seed=config.seed)
        folds = splitter.get_loso_folds()
        
        print(f"\n{'='*50}")
        print(f"  {name.upper()} — LOSO Folds ({len(folds)} folds)")
        print(f"{'='*50}")
        
        fold_data = []
        for i, fold in enumerate(folds):
            test_subj = fold["test"][0]
            train_subjs = fold["train"]
            
            test_seqs = [s for s in sequences if s.subject_id == test_subj]
            train_seqs = [s for s in sequences if s.subject_id in set(train_subjs)]
            
            test_falls = sum(1 for s in test_seqs if s.label == LABEL_FALL)
            test_adls = sum(1 for s in test_seqs if s.label == LABEL_ADL)
            train_falls = sum(1 for s in train_seqs if s.label == LABEL_FALL)
            train_adls = sum(1 for s in train_seqs if s.label == LABEL_ADL)
            
            fold_data.append({
                "fold": i,
                "test_subject": test_subj,
                "train_total": len(train_seqs),
                "train_falls": train_falls,
                "train_adls": train_adls,
                "test_total": len(test_seqs),
                "test_falls": test_falls,
                "test_adls": test_adls,
            })
            
            print(f"\n  Fold {i}: test={test_subj}")
            print(f"    Train: {len(train_seqs)} ({train_falls} falls, {train_adls} ADLs)")
            print(f"    Test:  {len(test_seqs)} ({test_falls} falls, {test_adls} ADLs)")
            
            if test_falls == 0:
                print(f"    WARNING: No fall sequences in test set!")
            if train_falls == 0:
                print(f"    WARNING: No fall sequences in training set!")
        
        # Save LOSO folds to data/splits/
        splits_dir = PROJECT_ROOT / "data" / "splits"
        splits_dir.mkdir(parents=True, exist_ok=True)
        splitter.save_splits(folds, splits_dir / f"{name}_loso_folds.json")
        print(f"\n  Folds saved to: data/splits/{name}_loso_folds.json")
else:
    print("No datasets loaded — skipping.")

## 5. Processed Keypoint Exploration

If preprocessing has been run (`python scripts/preprocess.py --dataset urfd`), explore the extracted keypoint quality: confidence distributions, NaN rates, and sequence statistics.

In [ ]:
def explore_processed(dataset_name: str):
    """Analyze processed keypoint data if available."""
    proc_dir = PROJECT_ROOT / "data" / "processed" / dataset_name
    metadata_path = proc_dir / "metadata.json"
    
    if not metadata_path.exists():
        print(f"  No processed data for {dataset_name}.")
        print(f"  Run: python scripts/preprocess.py --dataset {dataset_name}")
        return
    
    with open(metadata_path, "r") as f:
        meta = json.load(f)
    
    stats = meta["stats"]
    sequences = meta["sequences"]
    
    print(f"\n{'='*50}")
    print(f"  {dataset_name.upper()} — Processed Data")
    print(f"{'='*50}")
    print(f"  Processed sequences: {stats['processed']}")
    print(f"  Failed extractions:  {stats['failed']}")
    print(f"  Total frames:        {stats['total_frames']}")
    print(f"  NaN frames:          {stats['nan_frames']} ({stats['nan_frames']/max(stats['total_frames'],1):.1%})")
    
    # Sequence length distribution
    lengths = [s["num_frames"] for s in sequences]
    nan_counts = [s["nan_frames"] for s in sequences]
    
    if lengths:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        axes[0].hist(lengths, bins=20, color="#2196F3", edgecolor="black", linewidth=0.5)
        axes[0].set_xlabel("Number of frames")
        axes[0].set_ylabel("Count")
        axes[0].set_title(f"{dataset_name.upper()} — Sequence Length Distribution")
        axes[0].axvline(np.median(lengths), color="red", linestyle="--", label=f"Median: {np.median(lengths):.0f}")
        axes[0].legend()
        
        nan_ratios = [n / max(l, 1) for n, l in zip(nan_counts, lengths)]
        axes[1].hist(nan_ratios, bins=20, color="#FF9800", edgecolor="black", linewidth=0.5)
        axes[1].set_xlabel("NaN frame ratio")
        axes[1].set_ylabel("Count")
        axes[1].set_title(f"{dataset_name.upper()} — NaN Rate per Sequence")
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n  Sequence lengths: min={min(lengths)}, max={max(lengths)}, "
              f"median={np.median(lengths):.0f}, mean={np.mean(lengths):.1f}")
    
    # Sample a processed keypoint file for detailed inspection
    kp_dir = proc_dir / "keypoints_raw"
    npy_files = sorted(kp_dir.glob("*.npy")) if kp_dir.exists() else []
    
    if npy_files:
        sample = np.load(str(npy_files[0]))
        print(f"\n  Sample file: {npy_files[0].name}")
        print(f"  Shape: {sample.shape}  (frames, keypoints, coords)")
        
        # Visibility/confidence distribution
        if sample.shape[2] >= 4:
            confidences = sample[:, :, 3].flatten()
            confidences = confidences[~np.isnan(confidences)]
            
            if len(confidences) > 0:
                fig, ax = plt.subplots(figsize=(8, 3))
                ax.hist(confidences, bins=50, color="#9C27B0", edgecolor="black", linewidth=0.5)
                ax.set_xlabel("Visibility / Confidence score")
                ax.set_ylabel("Count")
                ax.set_title(f"Keypoint Confidence Distribution (sample: {npy_files[0].name})")
                ax.axvline(0.5, color="red", linestyle="--", label="Threshold (0.5)")
                ax.legend()
                plt.tight_layout()
                plt.show()

for ds_name in ["urfd", "le2i", "up_fall"]:
    explore_processed(ds_name)

## 6. Sample Skeleton Visualization

Visualize skeleton keypoints from a sample frame to verify pose estimation quality. Uses MediaPipe 33-keypoint connections.

In [ ]:
from src.pose.keypoints import MEDIAPIPE_SKELETON_CONNECTIONS

def plot_skeleton(keypoints_frame, ax, title="Skeleton"):
    """
    Plot a single frame's skeleton keypoints.
    
    Parameters
    ----------
    keypoints_frame : np.ndarray
        Shape (33, 4) with (x, y, z, visibility).
    ax : matplotlib axis
    title : str
    """
    # Filter valid keypoints
    valid = ~np.isnan(keypoints_frame[:, 0])
    
    # Plot keypoints
    x = keypoints_frame[valid, 0]
    y = keypoints_frame[valid, 1]
    conf = keypoints_frame[valid, 3] if keypoints_frame.shape[1] >= 4 else np.ones(len(x))
    
    ax.scatter(x, -y, c=conf, cmap="RdYlGn", s=30, vmin=0, vmax=1, zorder=5)
    
    # Plot skeleton connections
    for (i, j) in MEDIAPIPE_SKELETON_CONNECTIONS:
        if valid[i] and valid[j]:
            ax.plot(
                [keypoints_frame[i, 0], keypoints_frame[j, 0]],
                [-keypoints_frame[i, 1], -keypoints_frame[j, 1]],
                color="gray", linewidth=1.5, alpha=0.7,
            )
    
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("-y (inverted)")

# Try to find a processed keypoint file to visualize
sample_found = False
for ds_name in ["urfd", "le2i", "up_fall"]:
    kp_dir = PROJECT_ROOT / "data" / "processed" / ds_name / "keypoints_raw"
    if not kp_dir.exists():
        continue
    
    npy_files = sorted(kp_dir.glob("*.npy"))
    if not npy_files:
        continue
    
    sample = np.load(str(npy_files[0]))
    T = sample.shape[0]
    
    # Plot 4 evenly-spaced frames
    indices = np.linspace(0, T - 1, min(4, T), dtype=int)
    fig, axes = plt.subplots(1, len(indices), figsize=(4 * len(indices), 5))
    if len(indices) == 1:
        axes = [axes]
    
    for ax, idx in zip(axes, indices):
        plot_skeleton(sample[idx], ax, title=f"Frame {idx}")
    
    plt.suptitle(f"Sample Skeleton: {npy_files[0].stem}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    sample_found = True
    break

if not sample_found:
    print("No processed keypoint files found.")
    print("Run preprocessing first: python scripts/preprocess.py --dataset urfd")

## 7. Windowing Statistics

Preview the number of training windows that will be generated from the dataset with the configured window size and stride.

In [ ]:
window_size = config.get("features.window_size", 30)
window_stride = config.get("features.window_stride", 5)

print(f"Window size:  {window_size} frames ({window_size / config.get('video.target_fps', 15):.1f}s)")
print(f"Window stride: {window_stride} frames")
print()

for ds_name in ["urfd", "le2i", "up_fall"]:
    proc_dir = PROJECT_ROOT / "data" / "processed" / ds_name
    metadata_path = proc_dir / "metadata.json"
    
    if not metadata_path.exists():
        continue
    
    with open(metadata_path, "r") as f:
        meta = json.load(f)
    
    total_windows = 0
    fall_windows = 0
    adl_windows = 0
    
    for seq_meta in meta["sequences"]:
        T = seq_meta["num_frames"]
        T_eff = max(T, window_size)  # padded if shorter
        n_windows = max(1, (T_eff - window_size) // window_stride + 1)
        total_windows += n_windows
        
        if seq_meta["label"] == LABEL_FALL:
            fall_windows += n_windows
        else:
            adl_windows += n_windows
    
    print(f"{ds_name.upper()}:")
    print(f"  Estimated windows: {total_windows}")
    print(f"  Fall windows:      {fall_windows} ({fall_windows/max(total_windows,1):.1%})")
    print(f"  ADL windows:       {adl_windows} ({adl_windows/max(total_windows,1):.1%})")
    
    if total_windows > 0:
        imbalance_ratio = max(fall_windows, adl_windows) / max(min(fall_windows, adl_windows), 1)
        print(f"  Imbalance ratio:   {imbalance_ratio:.1f}:1")
        if imbalance_ratio > 3:
            print(f"  NOTE: Significant imbalance. Class weighting is mandatory (config: class_weight_auto=true)")
    print()

## Summary

This notebook provides the foundational data exploration for the fall detection system. Key findings to verify after running on actual data:

1. **Class balance** — URFD has roughly balanced fall/ADL ratio. Le2i leans heavily toward falls (143 vs 48). Automatic class weighting is mandatory.
2. **LOSO folds** — Each fold must have fall sequences in both train and test. With URFD's 5 subjects, each fold removes ~20% of data.
3. **Keypoint quality** — Monitor the NaN frame rate. If >10% of frames have missing keypoints, pose estimation parameters may need adjustment.
4. **Sequence lengths** — If sequences are much shorter than the window size (30 frames), padding will dominate and may degrade model quality.

**Next steps:** Run `python scripts/preprocess.py --dataset urfd` to extract keypoints, then re-run this notebook for full analysis.